# Przetwarzanie języka naturalnego — 3 laboratoria  
## Metody klasyczne i mały LLM (`gemma2:2b` przez Ollama)

Ten notebook zawiera laboratorium:

1. **Klasyczne PJN vs LLM**  
   NER, klasyfikacja, streszczanie, generowanie tekstu, analiza wydźwięku


## Wymagania

Zakładamy środowisko z Pythonem 3.10+ oraz lokalnie uruchomionym Ollama.

Model:
- `gemma2:2b`

Przykładowe uruchomienie modelu:
```bash
ollama pull gemma2:2b
ollama run gemma2:2b
```

Lub po prostu:
```bash
ollama serve
```

Notebook korzysta z endpointu:
- `http://localhost:11434/api/generate`

---

## Cele laboratoriów

Po wykonaniu ćwiczeń student powinien:
- rozumieć różnicę między podejściem klasycznym i LLM,
- potrafić przygotować prosty pipeline PJN,
- umieć sterować odpowiedzią modelu poprzez prompt,
- rozumieć ograniczenia małego modelu lokalnego,
- potrafić zbudować prosty system oparty na dokumentach.

In [ ]:
# Jeśli czegoś brakuje, odkomentuj i uruchom:
# !pip install pandas scikit-learn spacy requests matplotlib
# !python -m spacy download pl_core_news_sm

## Importy
Uzupełnij poniższą komórkę samodzielnie. W klasycznym ujęciu będą nam potrzebne biblioteki do:
- pracy z tabelami,
- klasyfikacji,
- podobieństwa tekstów,
- wywołania lokalnego modelu,
- wizualizacji wyników.

In [1]:
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import spacy

In [5]:
nlp_pl = spacy.load('pl_core_news_sm')
text ="Wojna w iranie niezbyt pozytywnie wplywa na ilosc paliwa które mogę wlać do swojego baku i przez to zgubiłem portfel bo był za lekki"
tokens_pl = nlp_pl(text)
lemmas_pl = [token.lemma_ for token in tokens_pl]
print(lemmas_pl)

['wojna', 'w', 'iranie', 'niezbyt', 'pozytywnie', 'wplywać', 'na', 'ilosc', 'paliwo', 'który', 'móc', 'wlać', 'do', 'swój', 'bak', 'i', 'przez', 'to', 'zgubić być', 'portfel', 'bo', 'być', 'za', 'lekki']


In [7]:
[(token.text, token.lemma_, token.pos_) for token in tokens_pl]

[('Wojna', 'wojna', 'NOUN'),
 ('w', 'w', 'ADP'),
 ('iranie', 'iranie', 'ADV'),
 ('niezbyt', 'niezbyt', 'ADV'),
 ('pozytywnie', 'pozytywnie', 'ADV'),
 ('wplywa', 'wplywać', 'VERB'),
 ('na', 'na', 'ADP'),
 ('ilosc', 'ilosc', 'NOUN'),
 ('paliwa', 'paliwo', 'NOUN'),
 ('które', 'który', 'DET'),
 ('mogę', 'móc', 'VERB'),
 ('wlać', 'wlać', 'VERB'),
 ('do', 'do', 'ADP'),
 ('swojego', 'swój', 'DET'),
 ('baku', 'bak', 'NOUN'),
 ('i', 'i', 'CCONJ'),
 ('przez', 'przez', 'ADP'),
 ('to', 'to', 'PRON'),
 ('zgubiłem', 'zgubić być', 'VERB'),
 ('portfel', 'portfel', 'NOUN'),
 ('bo', 'bo', 'SCONJ'),
 ('był', 'być', 'AUX'),
 ('za', 'za', 'PART'),
 ('lekki', 'lekki', 'ADJ')]

In [11]:
ner_result = nlp_pl("Wojna w Iranie niezbyt pozytywnie wpływa na ilość paliwa, które mogę wlać do swojego baku i przez to zgubiłem portfel, bo był za lekki. A Mickiewicz to przereklamowany poeata bo Juliusz Słowacki jest lepszy")
[(token.text, token.ent_type_) for token in ner_result]

[('Wojna', ''),
 ('w', ''),
 ('Iranie', 'placeName'),
 ('niezbyt', ''),
 ('pozytywnie', ''),
 ('wpływa', ''),
 ('na', ''),
 ('ilość', ''),
 ('paliwa', ''),
 (',', ''),
 ('które', ''),
 ('mogę', ''),
 ('wlać', ''),
 ('do', ''),
 ('swojego', ''),
 ('baku', ''),
 ('i', ''),
 ('przez', ''),
 ('to', ''),
 ('zgubiłem', ''),
 ('portfel', ''),
 (',', ''),
 ('bo', ''),
 ('był', ''),
 ('za', ''),
 ('lekki', ''),
 ('.', ''),
 ('A', ''),
 ('Mickiewicz', 'persName'),
 ('to', ''),
 ('przereklamowany', ''),
 ('poeata', ''),
 ('bo', ''),
 ('Juliusz', 'persName'),
 ('Słowacki', 'persName'),
 ('jest', ''),
 ('lepszy', '')]

## Funkcje pomocnicze

W kolejnej komórce przygotuj:
- funkcję do odpytywania Ollamy,
- funkcję pomocniczą do ładnego wypisywania JSON,
- prostą funkcję ewaluacyjną.

In [53]:
# TODO: przygotuj funkcję ask_llm(prompt, model='gemma2:2b')
# która wysyła zapytanie do Ollama API i zwraca tekst odpowiedzi

import requests as r

def ask_llm(prompt, model="gemma2:2b", temperature=0.2):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    return data["response"]


def print_json(text):
    try:
        parsed = json.loads(text)
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
    except Exception:
        print(text)

In [28]:
variablex = ask_llm("Tell me two interesting facts about the Alpacas")

In [29]:
variablex['response']

'Here are two interesting facts about alpacas:\n\n1. **Alpacas have a "language" of their own!**  They use a complex system of hums, whistles, and clicks to communicate with each other about everything from danger and predators to food sources and social status. \n2. **Alpacas can actually "spit" – but it\'s not just for anger.** They have special saliva in their mouths that helps them groom their thick fleece and also serves as a defense mechanism against predators.\n\nLet me know if you\'d like to learn more about these fascinating animals!  🦙 \n'

<details>
<summary><strong>Hint: funkcja do wywołania Ollamy</strong></summary>

```python
def ask_llm(prompt, model="gemma2:2b", temperature=0.2):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    return data["response"]


def print_json(text):
    try:
        parsed = json.loads(text)
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
    except Exception:
        print(text)
```
</details>

## Dane do ćwiczeń

Na potrzeby laboratoriów użyjemy małego, sztucznego zbioru danych, żeby łatwo porównywać wyniki.

W praktyce możesz potem podmienić dane na:
- komentarze z internetu,
- krótkie newsy,
- maile,
- opisy spraw,
- dokumenty urzędowe.

In [ ]:
# TODO: przygotuj DataFrame z kolumnami:
# text, label, sentiment
# label może oznaczać np. kategorię wiadomości: sport / technologia / biznes / kultura

<details>
<summary><strong>Hint: przykładowy zbiór danych</strong></summary>

```python
data = [
    {
        "text": "OpenAI zaprezentowało nowy model do analizy dokumentów i obrazów.",
        "label": "technologia",
        "sentiment": "neutralny"
    },
    {
        "text": "Legia wygrała mecz po emocjonującej końcówce i awansowała w tabeli.",
        "label": "sport",
        "sentiment": "pozytywny"
    },
    {
        "text": "Ceny energii znów rosną, co budzi niepokój przedsiębiorców.",
        "label": "biznes",
        "sentiment": "negatywny"
    },
    {
        "text": "W Krakowie otwarto wystawę poświęconą sztuce współczesnej.",
        "label": "kultura",
        "sentiment": "pozytywny"
    },
    {
        "text": "Nowy smartfon firmy X oferuje lepszy aparat i dłuższy czas pracy baterii.",
        "label": "technologia",
        "sentiment": "pozytywny"
    },
    {
        "text": "Reprezentacja Polski przegrała po słabym występie w drugiej połowie.",
        "label": "sport",
        "sentiment": "negatywny"
    },
    {
        "text": "Inflacja utrzymuje się na wysokim poziomie mimo wcześniejszych prognoz spadku.",
        "label": "biznes",
        "sentiment": "negatywny"
    },
    {
        "text": "Premiera nowego spektaklu zebrała bardzo dobre recenzje krytyków.",
        "label": "kultura",
        "sentiment": "pozytywny"
    },
    {
        "text": "Firma zapowiedziała redukcję zatrudnienia w kilku działach.",
        "label": "biznes",
        "sentiment": "negatywny"
    },
    {
        "text": "Badacze opracowali nową metodę trenowania małych modeli językowych.",
        "label": "technologia",
        "sentiment": "neutralny"
    },
    {
        "text": "Zawodnik pobił rekord życiowy i zdobył złoty medal.",
        "label": "sport",
        "sentiment": "pozytywny"
    },
    {
        "text": "Festiwal filmowy przyciągnął tłumy widzów i znanych reżyserów.",
        "label": "kultura",
        "sentiment": "pozytywny"
    }
]

df = pd.DataFrame(data)
df
```
</details>

In [30]:
data = [
    {
        "text": "OpenAI zaprezentowało nowy model do analizy dokumentów i obrazów.",
        "label": "technologia",
        "sentiment": "neutralny"
    },
    {
        "text": "Legia wygrała mecz po emocjonującej końcówce i awansowała w tabeli.",
        "label": "sport",
        "sentiment": "pozytywny"
    },
    {
        "text": "Ceny energii znów rosną, co budzi niepokój przedsiębiorców.",
        "label": "biznes",
        "sentiment": "negatywny"
    },
    {
        "text": "W Krakowie otwarto wystawę poświęconą sztuce współczesnej.",
        "label": "kultura",
        "sentiment": "pozytywny"
    },
    {
        "text": "Nowy smartfon firmy X oferuje lepszy aparat i dłuższy czas pracy baterii.",
        "label": "technologia",
        "sentiment": "pozytywny"
    },
    {
        "text": "Reprezentacja Polski przegrała po słabym występie w drugiej połowie.",
        "label": "sport",
        "sentiment": "negatywny"
    },
    {
        "text": "Inflacja utrzymuje się na wysokim poziomie mimo wcześniejszych prognoz spadku.",
        "label": "biznes",
        "sentiment": "negatywny"
    },
    {
        "text": "Premiera nowego spektaklu zebrała bardzo dobre recenzje krytyków.",
        "label": "kultura",
        "sentiment": "pozytywny"
    },
    {
        "text": "Firma zapowiedziała redukcję zatrudnienia w kilku działach.",
        "label": "biznes",
        "sentiment": "negatywny"
    },
    {
        "text": "Badacze opracowali nową metodę trenowania małych modeli językowych.",
        "label": "technologia",
        "sentiment": "neutralny"
    },
    {
        "text": "Zawodnik pobił rekord życiowy i zdobył złoty medal.",
        "label": "sport",
        "sentiment": "pozytywny"
    },
    {
        "text": "Festiwal filmowy przyciągnął tłumy widzów i znanych reżyserów.",
        "label": "kultura",
        "sentiment": "pozytywny"
    }
]

df = pd.DataFrame(data)
df

,text,label,sentiment
0,OpenAI zaprezentowało nowy model do analizy do...,technologia,neutralny
1,Legia wygrała mecz po emocjonującej końcówce i...,sport,pozytywny
2,"Ceny energii znów rosną, co budzi niepokój prz...",biznes,negatywny
3,W Krakowie otwarto wystawę poświęconą sztuce w...,kultura,pozytywny
4,Nowy smartfon firmy X oferuje lepszy aparat i ...,technologia,pozytywny
5,Reprezentacja Polski przegrała po słabym wystę...,sport,negatywny
6,Inflacja utrzymuje się na wysokim poziomie mim...,biznes,negatywny
7,Premiera nowego spektaklu zebrała bardzo dobre...,kultura,pozytywny
8,Firma zapowiedziała redukcję zatrudnienia w ki...,biznes,negatywny
9,Badacze opracowali nową metodę trenowania mały...,technologia,neutralny


## Klasyczne metody PJN vs mały LLM

W tym laboratorium porównasz dwa podejścia:
- klasyczne,
- oparte na LLM.

Zadania:
1. NER  
2. Klasyfikacja tekstu  
3. Streszczanie  
4. Generowanie tekstu  
5. Analiza wydźwięku

## 1. NER — rozpoznawanie encji nazwanych

NER, czyli Named Entity Recognition (rozpoznawanie encji nazwanych), to zadanie z zakresu NLP polegające na wykrywaniu w tekście fragmentów odnoszących się do określonych bytów świata rzeczywistego oraz przypisywaniu im kategorii. Najczęściej są to takie klasy jak osoba, organizacja, lokalizacja, data, kwota czy nazwa produktu. Przykładowo w zdaniu „Anna Nowak pracuje w Google w Warszawie” system NER powinien rozpoznać „Anna Nowak” jako osobę, „Google” jako organizację, a „Warszawie” jako lokalizację.

Zadanie to jest ważne, ponieważ pozwala przekształcać tekst nieustrukturyzowany w dane bardziej uporządkowane, które można dalej analizować automatycznie. NER znajduje zastosowanie między innymi w wyszukiwaniu informacji, analizie dokumentów, systemach pytanie–odpowiedź, ekstrakcji wiedzy, monitoringu mediów czy anonimizacji danych. W praktyce jest to jeden z podstawowych kroków umożliwiających zrozumienie, o kim, o czym, gdzie i kiedy mówi dany tekst.

### Zadanie
Dla podanego tekstu:
- wyodrębnij encje metodą klasyczną,
- wyodrębnij encje przy pomocy LLM,
- porównaj wyniki.

Najpierw przygotuj przykładowy tekst.

In [ ]:
# TODO: wpisz własny tekst z osobami, organizacjami, miejscami i datami
text_ner = "Magda Król prowadzi zajęcia na AGH w Krakowie. 15 marca 2026 spotkała się z zespołem OpenAI w Warszawie."
print(text_ner)

### Część klasyczna -- spaCy

[spaCy](http://spacy.io) to jedna z najpopularniejszych bibliotek do przetwarzania języka naturalnego w Pythonie, zaprojektowana przede wszystkim z myślą o zastosowaniach praktycznych i wydajnym działaniu na rzeczywistych danych. Umożliwia szybkie wykonywanie podstawowych zadań NLP, takich jak tokenizacja, analiza morfosyntaktyczna, lematyzacja, rozpoznawanie encji nazwanych (NER), tagowanie części mowy czy analiza zależności składniowych.

Dużą zaletą `spaCy` jest to, że oferuje gotowe modele językowe dla wielu języków, dzięki czemu można stosunkowo szybko rozpocząć pracę bez budowania całego pipeline’u od zera. Biblioteka działa na obiektach reprezentujących tekst, zdania i tokeny, co ułatwia dalszą analizę oraz integrację z kodem. W porównaniu do niektórych starszych narzędzi spaCy jest zwykle bardziej nowoczesne, szybsze i wygodniejsze w pracy inżynierskiej.

Załaduj polski model spaCy i pokaż znalezione encje.

In [35]:
# TODO: załaduj model spaCy, stokenizuj, zlematyzuj i przypisz interpretację morfologiczną elementom dowolnego zdania.
import spacy
nlp_pl = spacy.load('pl_core_news_sm')
df["nlp_pl"] = df["text"].apply(lambda x: nlp_pl(x))
df["tokens_pl"] = df["nlp_pl"].apply(lambda x: [token.text for token in x])
df["lemmas_pl"] = df["nlp_pl"].apply(lambda x: [token.lemma_ for token in x])
df["morph_pl"] = df["nlp_pl"].apply(lambda x: [token.morph for token in x])

In [38]:
df.lemmas_pl[0]

['openAI',
 'zaprezentować',
 'nowy',
 'model',
 'do',
 'analiza',
 'dokument',
 'i',
 'obraz',
 '.']

<details>
<summary><strong>Hint: NER w spaCy</strong></summary>

```python
nlp = spacy.load("pl_core_news_sm")
doc = nlp(text_ner)

for ent in doc.ents:
    print(ent.text, ent.label_)
```
</details>

### Część LLM
Poproś model o zwrócenie encji w formacie JSON.

Zadbaj o to, żeby:
- format był jednoznaczny,
- model nie dopisywał komentarzy,
- odpowiedź zawierała tylko JSON,
- sprawdź, czy język promptu ma znaczenie dla wyników.

In [40]:
text_ner = "Magda Król prowadzi zajęcia na AGH w Krakowie. 15 marca 2026 spotkała się z zespołem OpenAI w Warszawie."

In [ ]:
ze_prompt =  f'''
Wyodrębnij z tekstu encje nazwane.
Zwróć WYŁĄCZNIE poprawny JSON.
Format:
{{
  "persons": [],
  "organizations": [],
  "locations": [],
  "dates": []
}}

Tekst:
"""{text_ner}"""
'''

categories = ask_llm(ze_prompt)

TypeError: string indices must be integers, not 'str'

In [47]:
#jsonlike tu further cleanup i konwersja do słownika Pythona
categories

'```json\n{\n  "persons": [\n    "Magda Król"\n  ],\n  "organizations": [\n    "AGH",\n    "OpenAI"\n  ],\n  "locations": [\n    "Kraków",\n    "Warszawa"\n  ],\n  "dates": [\n    "15 marca 2026"\n  ]\n}\n```'

In [50]:
spicy = spacy.load('pl_core_news_sm')
doc = spicy(text_ner) 
for ent in doc.ents:
    print(ent.text, ent.lemma_, ent.label_)

Magda Król Magda Król persName
AGH AGH orgName
Krakowie Kraków placeName
15 marca 2026 15 marzec 2026 date
OpenAI OpenAI orgName
Warszawie Warszawa placeName


<details>
<summary><strong>Hint: prompt do NER z JSON</strong></summary>

```python
prompt = f'''
Wyodrębnij z tekstu encje nazwane.
Zwróć WYŁĄCZNIE poprawny JSON.
Format:
{{
  "persons": [],
  "organizations": [],
  "locations": [],
  "dates": []
}}

Tekst:
"""{text_ner}"""
'''

response = ask_llm(prompt)
print(response)
print_json(response)
```
</details>

### Pytania do omówienia
1. Czy model klasyczny i LLM wykryły te same encje?
2. Które encje były problematyczne?
3. Czy odpowiedź LLM była stabilna przy kilku uruchomieniach?

## 2. Klasyfikacja tekstu

Klasyfikacja tekstu to zadanie z zakresu NLP polegające na przypisaniu tekstowi jednej lub kilku kategorii na podstawie jego treści. Kategorie mogą dotyczyć bardzo różnych aspektów, na przykład tematu wiadomości, typu zgłoszenia, intencji użytkownika, rodzaju dokumentu albo wydźwięku emocjonalnego. Przykładowo krótki artykuł może zostać sklasyfikowany jako „sport”, „technologia” lub „kultura”, a wiadomość od klienta jako „reklamacja”, „pytanie” albo „prośba o kontakt”.

Jest to jedno z najczęściej wykorzystywanych zadań w praktycznych zastosowaniach NLP, ponieważ pozwala automatycznie porządkować duże zbiory tekstów i wspierać podejmowanie decyzji. Klasyfikacja tekstu znajduje zastosowanie między innymi w filtrowaniu spamu, analizie opinii, obsłudze klienta, moderacji treści, kategoryzacji dokumentów oraz systemach rekomendacyjnych.

W klasycznym podejściu klasyfikacja tekstu polega zwykle na zamianie tekstu na reprezentację numeryczną, na przykład z użyciem BoW lub TF-IDF, a następnie zastosowaniu modelu uczącego się, takiego jak Logistic Regression, Naive Bayes czy SVM. W podejściu opartym o LLM model może przypisywać etykiety bez dodatkowego trenowania albo na podstawie kilku przykładów umieszczonych w promptcie. Dzięki temu klasyfikacja tekstu jest dobrym zadaniem do porównywania metod klasycznych i współczesnych modeli językowych.

### Zadanie
Zbuduj klasyfikator tematów:
- wariant klasyczny: TF-IDF + Logistic Regression,
- wariant LLM: przypisanie etykiety przez prompt.

In [51]:
# TODO: podziel dane na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"]
)

<details>
<summary><strong>Hint: podział danych</strong></summary>

```python
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"]
)
```
</details>

In [ ]:
# TODO: stwórz klasyczny pipeline:
# 1. TfidfVectorizer
# 2. LogisticRegression
# 3. predykcja na zbiorze testowym

<details>
<summary><strong>Hint: klasyfikacja klasyczna</strong></summary>

```python
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

preds = clf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
```
</details>

In [52]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

preds = clf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.5
              precision    recall  f1-score   support

      biznes       0.00      0.00      0.00         1
     kultura       0.33      1.00      0.50         1
       sport       0.00      0.00      0.00         1
 technologia       1.00      1.00      1.00         1

    accuracy                           0.50         4
   macro avg       0.33      0.50      0.38         4
weighted avg       0.33      0.50      0.38         4



/home/matt/NLP OLLAMA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/matt/NLP OLLAMA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/matt/NLP OLLAMA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

### Wariant LLM
Dla każdego tekstu testowego poproś model o wybór jednej etykiety z listy:
- sport
- technologia
- biznes
- kultura

In [55]:
# TODO: przygotuj prompt klasyfikacyjny i wykonaj predykcję dla kilku przykładów
labels = ["sport", "technologia", "biznes", "kultura"]

def classify_with_llm(text):
    prompt = f'''
Przypisz tekst do dokładnie jednej kategorii:
{", ".join(labels)}

Zwróć wyłącznie nazwę kategorii.
Tekst:
"""{text}"""
'''
    return ask_llm(prompt).strip()

for text in X_test.tolist():
    print("TEKST:", text)
    print("LLM :", classify_with_llm(text))
    print("-" * 60)

TEKST: Firma zapowiedziała redukcję zatrudnienia w kilku działach.
LLM : biznes
------------------------------------------------------------
TEKST: W Krakowie otwarto wystawę poświęconą sztuce współczesnej.
LLM : Kultura
------------------------------------------------------------
TEKST: Zawodnik pobił rekord życiowy i zdobył złoty medal.
LLM : Sport
------------------------------------------------------------
TEKST: OpenAI zaprezentowało nowy model do analizy dokumentów i obrazów.
LLM : technologia
------------------------------------------------------------


<details>
<summary><strong>Hint: prompt do klasyfikacji</strong></summary>

```python
labels = ["sport", "technologia", "biznes", "kultura"]

def classify_with_llm(text):
    prompt = f'''
Przypisz tekst do dokładnie jednej kategorii:
{", ".join(labels)}

Zwróć wyłącznie nazwę kategorii.
Tekst:
"""{text}"""
'''
    return ask_llm(prompt).strip()

for text in X_test.tolist():
    print("TEKST:", text)
    print("LLM :", classify_with_llm(text))
    print("-" * 60)
```
</details>

### Zadanie z gwiazdką
Policz dokładność odpowiedzi LLM na zbiorze testowym i porównaj z modelem klasycznym.

In [ ]:
# TODO: przygotuj listę predykcji LLM i porównaj z y_test

## 3. Streszczanie

Streszczanie to zadanie z zakresu NLP polegające na skróceniu tekstu przy zachowaniu najważniejszych informacji w nim zawartych. Celem nie jest więc jedynie zmniejszenie objętości wypowiedzi, ale takie przetworzenie treści, aby odbiorca mógł szybko zrozumieć główny sens dokumentu, artykułu, raportu czy wiadomości. W praktyce streszczenie powinno pomijać szczegóły mniej istotne, a jednocześnie zachowywać kluczowe fakty, zależności i wnioski.

Wyróżnia się dwa podstawowe podejścia do streszczania. Pierwsze to streszczanie ekstrakcyjne, które polega na wyborze najważniejszych zdań lub fragmentów z oryginalnego tekstu. Drugie to streszczanie abstrakcyjne, w którym system sam formułuje nowe zdania opisujące sens tekstu, często w bardziej naturalny i spójny sposób. Metody klasyczne zwykle lepiej radzą sobie z podejściem ekstrakcyjnym, natomiast współczesne modele językowe umożliwiają również streszczanie abstrakcyjne.

Streszczanie ma bardzo szerokie zastosowanie, zwłaszcza tam, gdzie pracuje się z dużą liczbą dokumentów. Może wspierać analizę artykułów prasowych, dokumentacji, aktów prawnych, opinii klientów, korespondencji mailowej czy treści naukowych. Jest też dobrym przykładem zadania, w którym łatwo porównać klasyczne metody NLP z LLM, ponieważ modele językowe często tworzą streszczenia bardziej płynne i naturalne, ale jednocześnie mogą dopisywać informacje, których nie było w tekście źródłowym.

### Zadanie
Dla dłuższego tekstu przygotuj:
- streszczenie klasyczne,
- streszczenie wygenerowane przez LLM.

In [58]:
# TODO: wpisz dłuższy tekst do streszczenia
text_summary = '''
W ostatnich latach duże modele językowe stały się jednym z najważniejszych kierunków rozwoju
przetwarzania języka naturalnego. Ich zastosowania obejmują między innymi odpowiadanie na pytania,
streszczanie dokumentów, klasyfikację tekstu, generowanie kodu oraz ekstrakcję informacji.
Jednocześnie modele te nadal popełniają błędy, mogą konfabulować i bywają wrażliwe na sposób
sformułowania instrukcji. Z tego względu coraz częściej łączy się je z klasycznymi metodami
wyszukiwania informacji, regułami i narzędziami zewnętrznymi.
'''
print(text_summary)


W ostatnich latach duże modele językowe stały się jednym z najważniejszych kierunków rozwoju
przetwarzania języka naturalnego. Ich zastosowania obejmują między innymi odpowiadanie na pytania,
streszczanie dokumentów, klasyfikację tekstu, generowanie kodu oraz ekstrakcję informacji.
Jednocześnie modele te nadal popełniają błędy, mogą konfabulować i bywają wrażliwe na sposób
sformułowania instrukcji. Z tego względu coraz częściej łączy się je z klasycznymi metodami
wyszukiwania informacji, regułami i narzędziami zewnętrznymi.



### Wariant klasyczny
Zbuduj bardzo proste streszczanie ekstrakcyjne:
- podziel tekst na zdania,
- policz częstości słów lub TF-IDF,
- wybierz 1–2 najważniejsze zdania.

In [59]:
# TODO: napisz prosty algorytm streszczania ekstrakcyjnego
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text_summary.strip()) if s.strip()]

vectorizer_sum = TfidfVectorizer()
X_sum = vectorizer_sum.fit_transform(sentences)

sentence_scores = np.asarray(X_sum.sum(axis=1)).ravel()
best_idx = sentence_scores.argsort()[::-1][:2]

summary_classic = " ".join([sentences[i] for i in sorted(best_idx)])
print(summary_classic)

Ich zastosowania obejmują między innymi odpowiadanie na pytania,
streszczanie dokumentów, klasyfikację tekstu, generowanie kodu oraz ekstrakcję informacji. Z tego względu coraz częściej łączy się je z klasycznymi metodami
wyszukiwania informacji, regułami i narzędziami zewnętrznymi.


<details>
<summary><strong>Hint: bardzo prosty baseline streszczający</strong></summary>

```python
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text_summary.strip()) if s.strip()]

vectorizer_sum = TfidfVectorizer()
X_sum = vectorizer_sum.fit_transform(sentences)

sentence_scores = np.asarray(X_sum.sum(axis=1)).ravel()
best_idx = sentence_scores.argsort()[::-1][:2]

summary_classic = " ".join([sentences[i] for i in sorted(best_idx)])
print(summary_classic)
```
</details>

### Wariant LLM
Poproś model o krótkie streszczenie w 2 zdaniach.

In [60]:
# TODO: napisz prompt do streszczenia i wywołaj model
streszcz = ask_llm(f'''Strsc ponizszy tekst w dwoch zdaniach przy czym postaraj sie zachowac kazde waszne NER i sentyment - tekst to: {text_summary}''')
print(streszcz)

Wielkie modele językowe (LLM) odgrywają kluczową rolę w rozwoju przetwarzania języka naturalnego (NLP), oferując szerokie możliwości w dziedzinie odpowiadania na pytania, streszczania dokumentów, klasyfikacji tekstu i generowania kodu.  Natomiast ich ograniczenia, takie jak popełnianie błędów, konfabulowanie i wrażliwość na sposób sformułowania instrukcji, sprawiają, że coraz częściej łączą się z klasycznymi metodami wyszukiwania informacji. 



<details>
<summary><strong>Hint: prompt do streszczenia</strong></summary>

```python
prompt = f'''
Streść poniższy tekst w 2 zdaniach po polsku.
Nie dopisuj informacji spoza tekstu.

Tekst:
"""{text_summary}"""
'''

summary_llm = ask_llm(prompt)
print(summary_llm)
```
</details>

### Pytania do omówienia
1. Która wersja jest bardziej naturalna?
2. Czy LLM dopisał coś, czego nie było w tekście?
3. Czy baseline ekstrakcyjny zachował najważniejsze informacje?

## 4. Generowanie tekstu

Generowanie tekstu to zadanie z zakresu NLP polegające na tworzeniu nowej treści w języku naturalnym na podstawie podanego polecenia, kontekstu lub przykładu. Może to obejmować zarówno krótkie formy, takie jak tytuły, odpowiedzi, opisy czy maile, jak i dłuższe wypowiedzi, na przykład artykuły, streszczenia, dialogi lub fragmenty kodu. W odróżnieniu od klasyfikacji czy ekstrakcji informacji, tutaj system nie tylko analizuje istniejący tekst, ale sam konstruuje nową wypowiedź.

W klasycznych podejściach generowanie tekstu było zwykle oparte na szablonach, regułach albo prostych modelach statystycznych, takich jak modele n-gramowe. Pozwalało to tworzyć teksty poprawne tylko w ograniczonych, dobrze przewidzianych sytuacjach. Współczesne duże modele językowe znacznie rozszerzyły te możliwości, ponieważ potrafią generować treści bardziej płynne, spójne i dopasowane do stylu, tonu czy celu wypowiedzi.

Generowanie tekstu znajduje zastosowanie między innymi w automatycznym tworzeniu odpowiedzi, redagowaniu wiadomości, wspomaganiu pisania kodu, przygotowywaniu opisów produktów, tworzeniu materiałów marketingowych czy wspieraniu pracy z dokumentami. Jednocześnie jest to zadanie wymagające ostrożności, ponieważ model może generować treści brzmiące przekonująco, ale zawierające błędy, nieścisłości albo informacje nieobecne w danych wejściowych. Dlatego w praktyce generowanie tekstu często najlepiej sprawdza się nie jako całkowite zastąpienie człowieka, lecz jako narzędzie wspierające tworzenie i przekształcanie treści.

### Zadanie
Porównaj:
- prosty generator szablonowy,
- odpowiedź wygenerowaną przez LLM.

Temat: krótki formalny mail do studentów o zmianie terminu zajęć.

In [ ]:
# TODO: przygotuj prosty generator szablonowy

<details>
<summary><strong>Hint: generator szablonowy</strong></summary>

```python
topic = "zmiana terminu zajęć"
date = "18 kwietnia"
new_date = "25 kwietnia"

template_email = f'''
Szanowni Państwo,

informuję, że z powodu zmian organizacyjnych termin zajęć dotyczących: {topic}
został przeniesiony z dnia {date} na dzień {new_date}.

Pozdrawiam,
Prowadząca
'''.strip()

print(template_email)
```
</details>

In [ ]:
# TODO: wygeneruj analogiczny mail z pomocą LLM

<details>
<summary><strong>Hint: prompt do generowania maila</strong></summary>

```python
prompt = '''
Napisz krótki, formalny mail do studentów po polsku.
Temat: zmiana terminu zajęć.
Stary termin: 18 kwietnia.
Nowy termin: 25 kwietnia.
Styl: formalny, uprzejmy, zwięzły.
'''

email_llm = ask_llm(prompt)
print(email_llm)
```
</details>

## 5. Analiza wydźwięku

Analiza wydźwięku, nazywana też często analizą sentymentu, to zadanie z zakresu NLP polegające na określeniu, jaki emocjonalny lub oceniający charakter ma dany tekst. Najczęściej chodzi o przypisanie wypowiedzi do jednej z klas, takich jak pozytywny, neutralny lub negatywny, choć w bardziej zaawansowanych zastosowaniach można stosować dokładniejsze skale albo rozpoznawać konkretne emocje, takie jak złość, radość czy frustracja.

Zadanie to jest szczególnie przydatne wtedy, gdy chcemy automatycznie analizować duże zbiory opinii, komentarzy, recenzji, odpowiedzi ankietowych czy wiadomości od klientów. Dzięki temu można szybciej ocenić reakcje użytkowników na produkt, usługę, wydarzenie lub decyzję organizacji. Analiza wydźwięku jest więc szeroko wykorzystywana w marketingu, monitoringu mediów, badaniu opinii publicznej oraz systemach obsługi klienta.

W praktyce analiza wydźwięku bywa jednak trudniejsza, niż mogłoby się wydawać. Problem stanowią między innymi ironia, sarkazm, negacja, mieszane opinie oraz wypowiedzi zależne od kontekstu. Klasyczne podejścia często opierały się na słownikach słów pozytywnych i negatywnych albo na modelach klasyfikacyjnych uczonych na oznaczonych danych. Współczesne modele językowe potrafią lepiej uwzględniać kontekst całej wypowiedzi, ale nadal mogą mieć trudności w przypadkach niejednoznacznych. Dlatego analiza wydźwięku jest dobrym przykładem zadania, w którym można porównywać prostotę metod klasycznych z elastycznością LLM.

### Zadanie
Porównaj:
- prostą analizę opartą o słownik,
- analizę wykonaną przez LLM.

In [ ]:
# TODO: przygotuj bardzo prosty słownik pozytywnych i negatywnych słów

<details>
<summary><strong>Hint: prosty baseline sentymentu</strong></summary>

```python
positive_words = {"dobry", "świetny", "sukces", "awans", "złoty", "lepszy", "bardzo dobre"}
negative_words = {"zły", "słaby", "niepokój", "rosną", "przegrała", "redukcję", "wysokim"}

def sentiment_lexicon(text):
    t = text.lower()
    pos = sum(1 for w in positive_words if w in t)
    neg = sum(1 for w in negative_words if w in t)
    if pos > neg:
        return "pozytywny"
    elif neg > pos:
        return "negatywny"
    return "neutralny"

df["sentiment_classic"] = df["text"].apply(sentiment_lexicon)
df[["text", "sentiment", "sentiment_classic"]]
```
</details>

In [ ]:
# TODO: przygotuj funkcję sentiment_with_llm(text)

<details>
<summary><strong>Hint: prompt do sentymentu</strong></summary>

```python
def sentiment_with_llm(text):
    prompt = f'''
Oceń wydźwięk tekstu.
Dozwolone odpowiedzi: pozytywny, neutralny, negatywny.
Zwróć wyłącznie jedną etykietę.

Tekst:
"""{text}"""
'''
    return ask_llm(prompt).strip().lower()

df["sentiment_llm"] = df["text"].apply(sentiment_with_llm)
df[["text", "sentiment", "sentiment_classic", "sentiment_llm"]]
```
</details>

## Podsumowanie laboratorium 1

### Zadanie końcowe
Uzupełnij tabelę porównawczą:

| Zadanie | Metoda klasyczna | LLM | Co działa lepiej | Typowe błędy |
|---|---|---|---|---|

Następnie odpowiedz:
1. W którym zadaniu mały model lokalny działał najlepiej?
2. W którym zadaniu klasyczne metody były bardziej stabilne?
3. Jakie ograniczenia zauważyliście dla `gemma2:2b`?

<details>
<summary><strong>Bonus: prompty ulepszone</strong></summary>

```python
import json
import re

# 1. Wspólne helpery


def extract_json_block(text: str):
    """
    Próbuje wyciągnąć pierwszy blok JSON z odpowiedzi modelu.
    Przydatne, gdy model doda coś przed lub po JSON.
    """
    text = text.strip()
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return match.group(0)
    return text

def count_sentences(text: str) -> int:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)

def normalize_label(response: str, labels):
    response = response.strip().lower()
    if response in labels:
        return response
    for label in labels:
        if label in response:
            return label
    return "UNKNOWN"


# 2. NER

def ner_with_llm(text_ner: str):
    prompt = f'''
Jesteś systemem do ekstrakcji encji nazwanych.

Wyodrębnij z tekstu encje i zwróć je w kategoriach:
- persons
- organizations
- locations
- dates

Zasady:
- używaj tylko informacji z tekstu,
- nie zgaduj,
- zachowaj dokładny zapis z tekstu,
- jeśli brak encji, zwróć pustą listę,
- zwróć tylko poprawny JSON,
- nie dodawaj żadnego innego tekstu.

Format odpowiedzi:
{{
  "persons": [],
  "organizations": [],
  "locations": [],
  "dates": []
}}

Tekst:
{text_ner}
'''
    response = ask_llm(prompt)
    json_text = extract_json_block(response)

    try:
        parsed = json.loads(json_text)
    except Exception:
        parsed = {
            "persons": [],
            "organizations": [],
            "locations": [],
            "dates": []
        }

    for key in ["persons", "organizations", "locations", "dates"]:
        if key not in parsed or not isinstance(parsed[key], list):
            parsed[key] = []

    return parsed


# 3. Klasyfikacja tekstu


CLASS_LABELS = ["sport", "technologia", "biznes", "kultura"]

def classify_with_llm(text: str):
    prompt = f'''
Sklasyfikuj tekst do dokładnie jednej kategorii.

Kategorie:
- sport: sport, mecze, zawodnicy, wyniki, turnieje
- technologia: komputery, AI, internet, oprogramowanie, urządzenia
- biznes: firmy, finanse, rynek, inwestycje, sprzedaż
- kultura: film, muzyka, literatura, teatr, sztuka

Reguły:
- wybierz dokładnie jedną kategorię,
- użyj tylko jednej z podanych nazw,
- nie dodawaj wyjaśnień,
- nie dodawaj żadnego innego tekstu.

Tekst:
{text}

Kategoria:
'''
    response = ask_llm(prompt)
    return normalize_label(response, CLASS_LABELS)



# 4. Streszczenie

def summarize_with_llm(text_summary: str):
    prompt = f'''
Jesteś asystentem do streszczania tekstu.

Napisz streszczenie po polsku w dokładnie 2 zdaniach.

Zasady:
- używaj tylko informacji z tekstu,
- nie dopisuj nic spoza tekstu,
- uwzględnij główny temat i najważniejszy fakt lub wniosek,
- nie dodawaj komentarza, nagłówka ani wyjaśnienia,
- zwróć wyłącznie 2 zdania.

Tekst:
{text_summary}

Streszczenie:
'''
    response = ask_llm(prompt).strip()
    response = response.replace("Streszczenie:", "").strip()
    return response



# 5. Generowanie maila

def generate_email_with_llm(
    old_date="18 kwietnia",
    new_date="25 kwietnia"
):
    prompt = f'''
Jesteś asystentem do pisania formalnych maili.

Napisz krótki mail po polsku do studentów o zmianie terminu zajęć.

Dane:
- stary termin: {old_date}
- nowy termin: {new_date}

Wymagania:
- styl formalny, uprzejmy, zwięzły,
- treść ma mieć 3-5 zdań,
- uwzględnij obie daty,
- nie dopisuj przyczyny zmiany,
- nie dodawaj żadnego komentarza,
- zwróć tylko gotowy temat i treść maila.

Format:
Temat: ...
Treść:
Szanowni Państwo,
...
Z poważaniem,
Prowadząca
'''
    response = ask_llm(prompt).strip()
    return response



# 6. Sentyment

SENTIMENT_LABELS = ["pozytywny", "neutralny", "negatywny"]

def sentiment_with_llm(text: str):
    prompt = f'''
Sklasyfikuj sentyment tekstu.

Etykiety:
- pozytywny: pozytywny wydźwięk
- neutralny: głównie informacyjny lub bez wyraźnych emocji
- negatywny: negatywny wydźwięk

Zasady:
- wybierz dokładnie jedną etykietę,
- użyj tylko jednej z nazw: pozytywny, neutralny, negatywny,
- nie dodawaj wyjaśnień,
- nie dodawaj żadnego innego tekstu.

Tekst:
{text}

Sentyment:
'''
    response = ask_llm(prompt)
    return normalize_label(response, SENTIMENT_LABELS)
```
</details>